# Gradient term: prototype

The gradient term corrects each observation's dz and dy for the periscope
gradients OOBAGRD3 and OOBAGRD6. CALALIGN is unchanged; the term goes on top
of it.

Everything is in one file, `astrometry_corr_gradient/gradient_term.py`: the
code, the numbers and the test vectors. It needs only numpy and astropy.

**What to implement**, for each observation, on dz and dy:

`term = c3 * (OOBAGRD3 - ref3) + c6 * (OOBAGRD6 - ref6)`

1. **Gradients:** the observation's mean OOBAGRD3 and OOBAGRD6, in K, over its
   dwell (from the end of its maneuver to the start of the next), with
   telemetry in safing intervals masked.
2. **Coefficients:** c3 and c6 for the axis, below.
3. **References:** ref3 and ref6, from the reference table's row for the
   calendar quarter (UTC) of the observation's start time (DATE-OBS).
4. **Result:** the term in arcsec, the expected X-ray minus catalog offset. The
   corrected offset is the observed one minus the term.
5. **No term** if a gradient is missing or the quarter has no row: the term is
   0 and the observation is flagged.
6. **Check:** match the test vectors below.

**Still open:**

- How the term enters processing: which positions or alignment it corrects,
  and with which sign there.
- The averaging window: the dwell, as here, or the observation's GTI span.

Below: the coefficients, the reference table, the test vectors, then the whole
file. `gradient_term_validation.ipynb` shows that the term works.


In [1]:
import inspect
import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# this checkout's package, before any installed copy
sys.path.insert(0, str(Path.cwd().parent))
from astrometry_corr_gradient import gradient_term

## Coefficients

Two ways to write the same term:

- **Per K:** `term = c3 * (OOBAGRD3 - ref3) + c6 * (OOBAGRD6 - ref6)`
- **Per sigma:** `term = k3 * (OOBAGRD3 - ref3) / s3 + k6 * (OOBAGRD6 - ref6) / s6`

**ref3** and **ref6** are the quarter's references, from the table below.
**s** is how much each gradient typically moves inside a quarter. **k** = c x s
is the arcsec for that typical move, so the two gradients compare directly.


In [2]:
per_sigma = gradient_term.coefficients_per_sigma()
header = "| axis | gradient | c (arcsec per K) | s (K) | k (arcsec per sigma) |"
rows = [
    f"| {axis_name} | {name} | {gradient_term.COEFFICIENTS[axis_name][name]}"
    f" | {gradient_term.SIGMAS[name]} | {per_sigma[axis_name][name]:.4f} |"
    for axis_name in gradient_term.AXES
    for name in gradient_term.GRADIENT_MSIDS
]
display(Markdown("\n".join([header, "|---|---|---|---|---|", *rows])))
print("COEFFICIENTS =", gradient_term.COEFFICIENTS)
print("SIGMAS =", gradient_term.SIGMAS)
print(
    "per sigma =",
    {
        axis_name: {name: round(k, 4) for name, k in by_gradient.items()}
        for axis_name, by_gradient in per_sigma.items()
    },
)

| axis | gradient | c (arcsec per K) | s (K) | k (arcsec per sigma) |
|---|---|---|---|---|
| dz | OOBAGRD3 | 0.475 | 0.1117 | 0.0531 |
| dz | OOBAGRD6 | -13.581 | 0.00737 | -0.1001 |
| dy | OOBAGRD3 | 1.559 | 0.1117 | 0.1741 |
| dy | OOBAGRD6 | -11.532 | 0.00737 | -0.0850 |

COEFFICIENTS = {'dz': {'OOBAGRD3': 0.475, 'OOBAGRD6': -13.581}, 'dy': {'OOBAGRD3': 1.559, 'OOBAGRD6': -11.532}}
SIGMAS = {'OOBAGRD3': 0.1117, 'OOBAGRD6': 0.00737}
per sigma = {'dz': {'OOBAGRD3': 0.0531, 'OOBAGRD6': -0.1001}, 'dy': {'OOBAGRD3': 0.1741, 'OOBAGRD6': -0.085}}


## Reference table

One row per calendar quarter. Observations in that quarter use its OOBAGRD3
and OOBAGRD6, in K, as ref3 and ref6. `gradient_term()` only looks up the row;
how the row was made does not matter to it. `from_quarter` is the quarter
whose observations made the row; `observations` is how many.


In [3]:
gradient_term.reference_table().pprint(max_lines=-1, max_width=-1)

quarter   start    from_quarter observations OOBAGRD3 OOBAGRD6
                                                K        K    
------- ---------- ------------ ------------ -------- --------
 2004Q1 2004-01-01       2004Q1           52  0.85056  0.04103
 2004Q2 2004-04-01       2004Q2           39  0.85104  0.04103
 2004Q3 2004-07-01       2004Q3           34  0.83864  0.04103
 2004Q4 2004-10-01       2004Q4           54  0.82879  0.04103
 2005Q1 2005-01-01       2005Q1           60  0.83839  0.04107
 2005Q2 2005-04-01       2005Q2           58  0.83208  0.04236
 2005Q3 2005-07-01       2005Q3           36   0.8199  0.04219
 2005Q4 2005-10-01       2005Q4           61  0.81237  0.04169
 2006Q1 2006-01-01       2006Q1           65  0.77691  0.04416
 2006Q2 2006-04-01       2006Q2           59  0.78264  0.04416
 2006Q3 2006-07-01       2006Q3           46  0.77511  0.04416
 2006Q4 2006-10-01       2006Q4           47  0.73879  0.04593
 2007Q1 2007-01-01       2007Q1           52   0.6774  

## Test vectors

Real observations and the term each should get, in arcsec, next to what
`gradient_term()` gives. The last one has no telemetry, so it gets no term.
Any other implementation should match these too.


In [4]:
vectors = gradient_term.vector_table()
term = gradient_term.gradient_term(
    vectors["date"], {name: vectors[name] for name in gradient_term.GRADIENT_MSIDS}
)
for axis_name in gradient_term.AXES:
    vectors[f"{axis_name} computed"] = np.round(term[axis_name], 4)
    vectors[f"{axis_name} computed"].unit = "arcsec"
vectors.pprint(max_width=-1)

obsid         date        OOBAGRD3 OOBAGRD6    dz      dy     no_term    dz computed dy computed
                             K        K      arcsec  arcsec                 arcsec      arcsec  
----- ------------------- -------- -------- ------- ------- ------------ ----------- -----------
 7833 2008-04-01 04:58:27  0.70672  0.05355  0.0132  0.0433            -      0.0132      0.0433
13977 2012-07-01 02:51:34  0.43824  0.06293  0.0993  0.2223            -      0.0993      0.2223
19888 2016-10-01 15:29:43 -0.10007  0.07312  0.1502  0.2627            -      0.1502      0.2627
27455 2022-10-02 05:54:37 -0.51387  0.12805 -0.2392 -0.4389            -     -0.2392     -0.4389
30979 2025-07-02 12:02:23 -0.26719  0.14234 -0.0174  0.0513            -     -0.0174      0.0513
30432 2026-10-02 09:37:21 -0.38185  0.14024 -0.0484 -0.0833            -     -0.0484     -0.0833
11749 2010-05-29 20:41:42      nan      nan     0.0     0.0 no telemetry         0.0         0.0


## The whole file

`astrometry_corr_gradient/gradient_term.py`. Its docstring, at the top, says
what to implement; the numbers are at the end. After a new calibration set,
`python scripts/update_gradient_term_data.py` rewrites them.


In [5]:
print(inspect.getsource(gradient_term))

"""The gradient term: what processing needs.

The gradient term corrects each observation's dz and dy for the periscope
gradients OOBAGRD3 and OOBAGRD6. CALALIGN is unchanged; the term goes on top
of it.

What to implement, for each observation, on each axis (dz and dy)::

    term = c3 * (OOBAGRD3 - ref3) + c6 * (OOBAGRD6 - ref6)

- OOBAGRD3, OOBAGRD6: the observation's mean of each gradient, in K, over its
  dwell (from the end of its maneuver to the start of the next), with
  telemetry in safing intervals masked. We average cheta's 5-minute means.
- c3, c6: ``COEFFICIENTS``, arcsec per K. One pair per axis for the whole
  mission.
- ref3, ref6: the row of ``reference_table()`` for the calendar quarter (UTC)
  of the observation's start time (DATE-OBS).
- The term is in arcsec: the expected X-ray minus catalog offset. The
  corrected offset is the observed one minus the term.
- No term if a gradient is missing or the table has no row for the quarter:
  the term is 0 and the observati